### **Adversarial Attacks: Evasion and Robustness Testing**

> An adversarial example is an input changed slightly and deliberately so that a model makes the wrong decision. In security models this is called **evasion**: a fraudster adjusts transaction features just enough to look legitimate, or malware is modified to escape a classifier. This notebook trains a small fraud-like classifier from scratch with NumPy, attacks it with the **Fast Gradient Sign Method (FGSM)**, compares the attack with random noise of the same size, and measures how **adversarial training** improves robustness.

**Related frameworks:** MITRE ATLAS (evasion of ML models) · NIST AI 100-2 (adversarial machine learning taxonomy) · OWASP AI Testing Guide

| # | Pattern to review | Synthetic example | Defensive check or expected handling |
| ---: | --- | --- | --- |
| 1 | Gradient-based evasion | Small FGSM perturbation flips a fraud prediction | Measure accuracy under attack, not only on clean data |
| 2 | Random noise vs. crafted noise | Same perturbation size, very different impact | Compare against a random baseline |
| 3 | Feature manipulation | Amount split into smaller transactions | Use features the attacker cannot easily control |
| 4 | Out-of-range input | Negative amount or impossible time of day | Validate input ranges and schemas |
| 5 | Query probing | Many near-identical requests to find the boundary | Rate-limit and monitor query patterns |
| 6 | Transfer attack | Attack built on a surrogate model works on the target | Do not rely on secrecy of the model |
| 7 | Overconfident model | High score far from the training data | Calibrate scores and flag out-of-distribution inputs |
| 8 | Single-model decision | One score decides alone | Combine models, rules, and human review for high risk |
| 9 | No robustness metric | Only clean accuracy is reported | Report a robustness curve across perturbation sizes |
| 10 | Adversarial training | Train with attacked examples | Re-evaluate clean accuracy and robustness together |

In [1]:
#Import Libs
import numpy as np
rng = np.random.default_rng(42)

In [2]:
#Create a Synthetic, Fraud-like Dataset (classes 0 = legitimate, 1 = fraud)
#feature_1: very predictive but small-scale and EASY for an attacker to change (e.g., time between purchases)
#feature_2: noisier but HARD to fake (e.g., account and device history)
n = 400
y = np.array([0] * n + [1] * n)
feature_1 = np.where(y == 1, 0.4, -0.4) + rng.normal(0, 0.15, 2 * n)
feature_2 = np.where(y == 1, 1.2, -1.2) + rng.normal(0, 1.0, 2 * n)
X = np.c_[feature_1, feature_2]
idx = rng.permutation(len(y))
X, y = X[idx], y[idx]
X_train, y_train, X_test, y_test = X[:600], y[:600], X[600:], y[600:]
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (600, 2) Test: (200, 2)


In [3]:
#Function to Train a Logistic Regression (gradient descent; optional adversarial training)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def fgsm(w, b, X, y, epsilon):
    #FGSM: move each input by epsilon in the sign of the loss gradient w.r.t. the input
    p = sigmoid(X @ w + b)
    grad_x = (p - y)[:, None] * w[None, :]
    return X + epsilon * np.sign(grad_x)

def train_logistic(X, y, epochs=2000, lr=0.5, adversarial_epsilon=0.0):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(epochs):
        X_batch = fgsm(w, b, X, y, adversarial_epsilon) if adversarial_epsilon else X
        p = sigmoid(X_batch @ w + b)
        w -= lr * X_batch.T @ (p - y) / len(y)
        b -= lr * np.mean(p - y)
    return w, b

def accuracy(w, b, X, y):
    return np.mean((sigmoid(X @ w + b) >= 0.5) == y)

w, b = train_logistic(X_train, y_train)
print("Weights (feature_1, feature_2):", np.round(w, 2))
print("Clean test accuracy:", round(accuracy(w, b, X_test, y_test), 3))

Weights (feature_1, feature_2): [11.42  1.26]
Clean test accuracy: 0.995


### **01. FGSM Attack vs. Random Noise**

For logistic regression, the gradient of the loss with respect to the input is `(p - y) * w`. FGSM moves each input by `epsilon` in the sign of that gradient, the direction that hurts the model most.

In [4]:
#Function to Add Random Noise of the Same Size
def random_noise(X, epsilon):
    return X + epsilon * rng.choice([-1, 1], size=X.shape)

EPSILONS = [0.0, 0.2, 0.4, 0.6, 0.8]
print(f"{'epsilon':>8} | {'random noise':>12} | {'FGSM attack':>11}")
for eps in EPSILONS:
    acc_noise = accuracy(w, b, random_noise(X_test, eps), y_test)
    acc_fgsm = accuracy(w, b, fgsm(w, b, X_test, y_test, eps), y_test)
    print(f"{eps:>8} | {acc_noise:>12.3f} | {acc_fgsm:>11.3f}")

 epsilon | random noise | FGSM attack
     0.0 |        0.995 |       0.995
     0.2 |        0.965 |       0.955
     0.4 |        0.865 |       0.665
     0.6 |        0.635 |       0.245
     0.8 |        0.470 |       0.010


### **02. Example: One Fraud Case Evading the Model**

In [5]:
#Show a Fraud Example Before and After the Attack
fraud_cases = X_test[y_test == 1]
case = fraud_cases[np.argmin(np.abs(fraud_cases @ w + b - 1.0))][None, :]  # a fraud case with a moderate score
for eps in [0.0, 0.2, 0.4]:
    adv = fgsm(w, b, case, np.array([1]), eps)
    score = sigmoid(adv @ w + b)[0]
    print(f"epsilon={eps}: features={np.round(adv[0], 2)} fraud score={score:.3f} -> {'FRAUD' if score >= 0.5 else 'LEGITIMATE (evaded)'}")

epsilon=0.0: features=[-0.04  0.95] fraud score=0.722 -> FRAUD
epsilon=0.2: features=[-0.24  0.75] fraud score=0.171 -> LEGITIMATE (evaded)
epsilon=0.4: features=[-0.44  0.55] fraud score=0.016 -> LEGITIMATE (evaded)


### **03. Defense: Adversarial Training**

During training, every gradient step uses FGSM versions of the examples. The model learns to stop relying on the fragile, attacker-controllable feature: it gives up some clean accuracy and gains a lot of robustness.

In [6]:
#Train with Adversarial Examples (epsilon = 0.4)
w_rob, b_rob = train_logistic(X_train, y_train, adversarial_epsilon=0.4)
print("Standard weights   (feature_1, feature_2):", np.round(w, 2))
print("Adversarial weights (feature_1, feature_2):", np.round(w_rob, 2))
print()
print(f"{'epsilon':>8} | {'standard model':>14} | {'adversarially trained':>21}")
for eps in EPSILONS:
    a_std = accuracy(w, b, fgsm(w, b, X_test, y_test, eps), y_test)
    a_rob = accuracy(w_rob, b_rob, fgsm(w_rob, b_rob, X_test, y_test, eps), y_test)
    print(f"{eps:>8} | {a_std:>14.3f} | {a_rob:>21.3f}")

Standard weights   (feature_1, feature_2): [11.42  1.26]
Adversarial weights (feature_1, feature_2): [0.05 1.42]

 epsilon | standard model | adversarially trained
     0.0 |          0.995 |                 0.885
     0.2 |          0.955 |                 0.835
     0.4 |          0.665 |                 0.795
     0.6 |          0.245 |                 0.765
     0.8 |          0.010 |                 0.685


### **04. Reading the Results**

* The standard model is excellent on clean data, but it depends heavily on `feature_1`, which an attacker can shift cheaply. A perturbation of `0.6` drops its accuracy far below chance.
* Random noise of the same size hurts much less than the crafted attack (at `epsilon = 0.6`, about 0.64 against 0.25 accuracy): **robustness must be tested with crafted attacks, not only with noise**.
* The adversarially trained model almost ignores `feature_1`. It is less accurate on clean data, but far more stable under attack. In fraud detection, this is the trade-off between a model that fits history and a model that holds up against an adaptive adversary.

### **Limitations and Next Steps**

* This is a two-feature linear model; deep models are more fragile and need dedicated tools (for example, the Adversarial Robustness Toolbox).
* FGSM assumes the attacker knows the model. Real attackers often train a surrogate and **transfer** the attack, which is why model secrecy is not a defense.
* For fraud models, prefer features the attacker cannot control, combine models with rules and human review, and track robustness metrics alongside accuracy.